In [ ]:
"""
Hybrid Conversational Process Intelligence Prototype

Implementation of the hybrid experimental condition described in the paper
"From Event Logs to Conversational Process Intelligence: A Hybrid Architecture
and Prototype for Natural Language-Based Process Analysis."

Processing flow:
    User query
        -> Direct deterministic Python response, when applicable
        -> Otherwise, rule-based analytical capability selection
        -> Deterministic process analytics
        -> Structured analytical evidence
        -> LLM interpretation

The LLM does not receive direct access to the raw event log.
"""


# =============================================================================
# Dependencies and imports
# =============================================================================

!pip install -q pandas openpyxl google-genai

import pandas as pd
import json
import re
import unicodedata
import time

from datetime import datetime
from getpass import getpass

from google import genai
from google.genai import types

import ipywidgets as widgets
from IPython.display import display, Markdown


# =============================================================================
# Experiment configuration
# =============================================================================

EVENT_LOG_FILE = "EC 1 - Purchasing.xlsx"

GEMINI_MODEL = "gemini-2.5-flash-lite"

TEMPERATURE = 0.0

RESULTS_FILE = "hybrid_prototype_results.csv"

CONFIGURATION_FILE = "hybrid_prototype_configuration.json"

MINIMUM_LLM_INTERVAL = 60

TOP_N_VARIANTS = 10

TOP_N_WAITING_TIMES = 10


# Operational rework rule

REWORK_ACTIVITIES = [
    "Amend Request for Quotation",
    "Amend Purchase Requisition"
]


# =============================================================================
# Text utilities
# =============================================================================

def normalize_text(text):
    """
    Normalizes text before applying routing rules.
    """

    text = str(text).lower().strip()

    text = unicodedata.normalize(
        "NFD",
        text
    )

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Mn"
    )

    return text


def find_matches(text, patterns):

    matches = [
        pattern
        for pattern in patterns
        if pattern in text
    ]

    return matches


# =============================================================================
# Event log loading and validation
# =============================================================================

preprocessing_start = time.perf_counter()

df = pd.read_excel(
    EVENT_LOG_FILE
)


required_columns = [
    "Case ID",
    "Activity",
    "Start Timestamp",
    "Complete Timestamp"
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:

    raise ValueError(
        "Required columns missing from the event log: "
        + ", ".join(missing_columns)
    )


df["Start Timestamp"] = pd.to_datetime(
    df["Start Timestamp"],
    errors="coerce"
)


df["Complete Timestamp"] = pd.to_datetime(
    df["Complete Timestamp"],
    errors="coerce"
)


df = (
    df
    .sort_values(
        by=[
            "Case ID",
            "Start Timestamp"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)


missing_start_timestamps = (
    df["Start Timestamp"]
    .isna()
    .sum()
)


missing_complete_timestamps = (
    df["Complete Timestamp"]
    .isna()
    .sum()
)


# =============================================================================
# General process information
# =============================================================================

number_of_events = len(df)

number_of_cases = (
    df["Case ID"]
    .nunique()
)

number_of_activities = (
    df["Activity"]
    .nunique()
)

average_events_per_case = (
    number_of_events
    /
    number_of_cases
)

event_log_start = (
    df["Start Timestamp"]
    .min()
)

event_log_end = (
    df["Complete Timestamp"]
    .max()
)


print("\n==========================================")
print("EVENT LOG LOADED")
print("==========================================")

print(
    "Events:",
    number_of_events
)

print(
    "Cases:",
    number_of_cases
)

print(
    "Distinct activities:",
    number_of_activities
)

print(
    "Average events per case:",
    round(
        average_events_per_case,
        2
    )
)

print(
    "Recorded period:",
    event_log_start,
    "to",
    event_log_end
)

print(
    "Missing Start Timestamps:",
    missing_start_timestamps
)

print(
    "Missing Complete Timestamps:",
    missing_complete_timestamps
)

print("==========================================\n")


# =============================================================================
# Trace and variant construction
# =============================================================================

traces = (
    df
    .groupby(
        "Case ID",
        sort=False
    )["Activity"]
    .apply(list)
    .reset_index()
)


traces.columns = [
    "Case ID",
    "trace"
]


traces["variant"] = (
    traces["trace"]
    .apply(
        lambda activities:
        " → ".join(activities)
    )
)


traces["final_activity"] = (
    traces["trace"]
    .apply(
        lambda activities:
        activities[-1]
    )
)


variant_frequency = (
    traces["variant"]
    .value_counts()
    .reset_index()
)


variant_frequency.columns = [
    "variant",
    "case_count"
]


variant_frequency["variant_id"] = [
    f"V{i + 1}"
    for i in range(
        len(variant_frequency)
    )
]


number_of_variants = (
    len(
        variant_frequency
    )
)


# =============================================================================
# Case duration
# =============================================================================

case_duration = (
    df
    .groupby("Case ID")
    .agg(
        start=(
            "Start Timestamp",
            "min"
        ),
        end=(
            "Complete Timestamp",
            "max"
        )
    )
    .reset_index()
)


case_duration[
    "duration_hours"
] = (
    (
        case_duration["end"]
        -
        case_duration["start"]
    )
    .dt
    .total_seconds()
    /
    3600
)


case_duration[
    "duration_days"
] = (
    case_duration[
        "duration_hours"
    ]
    /
    24
)


cases_with_valid_duration = int(
    case_duration[
        "duration_hours"
    ]
    .notna()
    .sum()
)


cases_without_valid_duration = int(
    case_duration[
        "duration_hours"
    ]
    .isna()
    .sum()
)


# =============================================================================
# Case endings
# =============================================================================

def classify_status(final_activity):

    if final_activity == "Pay Invoice":
        return "Completed"

    if (
        final_activity
        ==
        "Analyze Request for Quotation"
    ):
        return "Terminated at RFQ"

    if (
        final_activity
        ==
        "Analyze Purchase Requisition"
    ):
        return "Terminated at Purchase Requisition"

    return "Other"


traces["status"] = (
    traces[
        "final_activity"
    ]
    .apply(
        classify_status
    )
)


# =============================================================================
# Rework operationalization
# =============================================================================

def count_rework(trace):

    return sum(
        trace.count(activity)
        for activity
        in REWORK_ACTIVITIES
    )


def classify_rework(trace):

    quantity = (
        count_rework(trace)
    )

    if quantity == 0:
        return "No rework"

    if quantity <= 2:
        return "Moderate rework"

    return "Intensive rework"


traces[
    "amend_rfq_count"
] = (
    traces["trace"]
    .apply(
        lambda activities:
        activities.count(
            "Amend Request for Quotation"
        )
    )
)


traces[
    "amend_pr_count"
] = (
    traces["trace"]
    .apply(
        lambda activities:
        activities.count(
            "Amend Purchase Requisition"
        )
    )
)


traces[
    "total_rework_count"
] = (
    traces["trace"]
    .apply(
        count_rework
    )
)


traces[
    "rework_group"
] = (
    traces["trace"]
    .apply(
        classify_rework
    )
)


# =============================================================================
# Case-level analytical base
# =============================================================================

case_base = (
    case_duration
    .merge(
        traces[
            [
                "Case ID",
                "variant",
                "final_activity",
                "status",
                "amend_rfq_count",
                "amend_pr_count",
                "total_rework_count",
                "rework_group"
            ]
        ],
        on="Case ID"
    )
    .merge(
        variant_frequency[
            [
                "variant",
                "variant_id"
            ]
        ],
        on="variant"
    )
)


# =============================================================================
# Activity frequency
# =============================================================================

activity_frequency = (
    df["Activity"]
    .value_counts()
    .rename_axis(
        "activity"
    )
    .reset_index(
        name="event_count"
    )
)


# =============================================================================
# Variant analysis
# =============================================================================

top_variants = (
    variant_frequency
    .head(
        TOP_N_VARIANTS
    )
    .copy()
)


top_variants_analysis = (
    case_base[
        case_base[
            "variant_id"
        ]
        .isin(
            top_variants[
                "variant_id"
            ]
        )
    ]
    .groupby(
        [
            "variant_id",
            "variant"
        ]
    )
    .agg(
        case_count=(
            "Case ID",
            "count"
        ),
        median_hours=(
            "duration_hours",
            "median"
        ),
        median_days=(
            "duration_days",
            "median"
        ),
        min_hours=(
            "duration_hours",
            "min"
        ),
        max_hours=(
            "duration_hours",
            "max"
        )
    )
    .reset_index()
)


variant_order = {
    f"V{i}": i
    for i in range(
        1,
        TOP_N_VARIANTS + 1
    )
}


top_variants_analysis[
    "order"
] = (
    top_variants_analysis[
        "variant_id"
    ]
    .map(
        variant_order
    )
)


top_variants_analysis = (
    top_variants_analysis
    .sort_values(
        "order"
    )
    .drop(
        columns="order"
    )
    .reset_index(
        drop=True
    )
)


for column in [
    "median_hours",
    "median_days",
    "min_hours",
    "max_hours"
]:

    top_variants_analysis[
        column
    ] = (
        top_variants_analysis[
            column
        ]
        .round(2)
    )


# =============================================================================
# Waiting-time analysis
# =============================================================================

events = (
    df.copy()
)


events[
    "next_activity"
] = (
    events
    .groupby(
        "Case ID"
    )[
        "Activity"
    ]
    .shift(-1)
)


events[
    "next_activity_start"
] = (
    events
    .groupby(
        "Case ID"
    )[
        "Start Timestamp"
    ]
    .shift(-1)
)


events[
    "waiting_time_hours"
] = (
    (
        events[
            "next_activity_start"
        ]
        -
        events[
            "Complete Timestamp"
        ]
    )
    .dt
    .total_seconds()
    /
    3600
)


events[
    "waiting_time_days"
] = (
    events[
        "waiting_time_hours"
    ]
    /
    24
)


intervals = (
    events
    .dropna(
        subset=[
            "waiting_time_hours"
        ]
    )
    .copy()
)


temporal_overlaps = (
    intervals[
        intervals[
            "waiting_time_hours"
        ]
        < 0
    ]
    .copy()
)


number_of_overlaps = (
    len(
        temporal_overlaps
    )
)


valid_waiting_times = (
    intervals[
        intervals[
            "waiting_time_hours"
        ]
        >= 0
    ]
    .copy()
)


waiting_time_columns = [
    "Case ID",
    "Activity",
    "next_activity",
    "waiting_time_hours",
    "waiting_time_days"
]


for optional_column in [
    "Resource",
    "Role"
]:

    if (
        optional_column
        in df.columns
    ):

        waiting_time_columns.append(
            optional_column
        )


longest_waiting_times = (
    valid_waiting_times
    .sort_values(
        by="waiting_time_hours",
        ascending=False
    )
    .head(
        TOP_N_WAITING_TIMES
    )[
        waiting_time_columns
    ]
    .copy()
)


longest_waiting_times[
    "waiting_time_hours"
] = (
    longest_waiting_times[
        "waiting_time_hours"
    ]
    .round(2)
)


longest_waiting_times[
    "waiting_time_days"
] = (
    longest_waiting_times[
        "waiting_time_days"
    ]
    .round(2)
)


# =============================================================================
# Analytical summaries
# =============================================================================

rework_analysis = (
    case_base
    .groupby(
        "rework_group"
    )
    .agg(
        case_count=(
            "Case ID",
            "count"
        ),
        median_hours=(
            "duration_hours",
            "median"
        ),
        median_days=(
            "duration_days",
            "median"
        ),
        min_hours=(
            "duration_hours",
            "min"
        ),
        max_hours=(
            "duration_hours",
            "max"
        )
    )
    .reset_index()
)


status_analysis = (
    case_base
    .groupby(
        "status"
    )
    .agg(
        case_count=(
            "Case ID",
            "count"
        ),
        median_hours=(
            "duration_hours",
            "median"
        ),
        median_days=(
            "duration_days",
            "median"
        ),
        min_hours=(
            "duration_hours",
            "min"
        ),
        max_hours=(
            "duration_hours",
            "max"
        )
    )
    .reset_index()
)


for table in [
    rework_analysis,
    status_analysis
]:

    for column in [
        "median_hours",
        "median_days",
        "min_hours",
        "max_hours"
    ]:

        table[
            column
        ] = (
            table[
                column
            ]
            .round(2)
        )


case_endings = (
    traces[
        "final_activity"
    ]
    .value_counts()
    .to_dict()
)


# =============================================================================
# Deterministic analytical capabilities
# =============================================================================

def analyze_general_information():

    return {
        "number_of_cases":
            int(
                number_of_cases
            ),

        "number_of_events":
            int(
                number_of_events
            ),

        "number_of_distinct_activities":
            int(
                number_of_activities
            ),

        "number_of_variants":
            int(
                number_of_variants
            ),

        "average_events_per_case":
            round(
                average_events_per_case,
                2
            )
    }


def analyze_activity_frequency():

    return {
        "number_of_distinct_activities":
            int(
                number_of_activities
            ),

        "activity_frequency":
            activity_frequency
            .to_dict(
                orient="records"
            )
    }


def analyze_case_duration():

    duration_series = (
        case_base[
            "duration_hours"
        ]
        .dropna()
    )

    return {
        "cases_with_valid_duration":
            int(
                cases_with_valid_duration
            ),

        "cases_without_valid_duration":
            int(
                cases_without_valid_duration
            ),

        "mean_hours":
            round(
                float(
                    duration_series.mean()
                ),
                2
            ),

        "mean_days":
            round(
                float(
                    duration_series.mean()
                    /
                    24
                ),
                2
            ),

        "median_hours":
            round(
                float(
                    duration_series.median()
                ),
                2
            ),

        "median_days":
            round(
                float(
                    duration_series.median()
                    /
                    24
                ),
                2
            ),

        "min_hours":
            round(
                float(
                    duration_series.min()
                ),
                2
            ),

        "max_hours":
            round(
                float(
                    duration_series.max()
                ),
                2
            )
    }


def analyze_variants():

    return {
        "total_number_of_variants":
            int(
                number_of_variants
            ),

        "number_of_variants_displayed":
            int(
                len(
                    top_variants_analysis
                )
            ),

        "top_variants":
            top_variants_analysis
            .to_dict(
                orient="records"
            )
    }


def analyze_rework():

    return {
        "rework_definition": {
            "activities_considered":
                REWORK_ACTIVITIES,

            "no_rework":
                "0 occurrences",

            "moderate_rework":
                "1 or 2 occurrences",

            "intensive_rework":
                "3 or more occurrences"
        },

        "results":
            rework_analysis
            .to_dict(
                orient="records"
            )
    }


def analyze_case_endings():

    return {
        "status_rule": {
            "Completed":
                "Final activity = Pay Invoice",

            "Terminated at RFQ":
                (
                    "Final activity = "
                    "Analyze Request for Quotation"
                ),

            "Terminated at Purchase Requisition":
                (
                    "Final activity = "
                    "Analyze Purchase Requisition"
                ),

            "Other":
                (
                    "Any other final activity"
                )
        },

        "final_activity":
            case_endings,

        "case_status":
            status_analysis
            .to_dict(
                orient="records"
            )
    }


def analyze_waiting_times():

    return {
        "definition":
            (
                "Time interval between the "
                "Complete Timestamp of an activity "
                "and the Start Timestamp of the "
                "next activity in the same case."
            ),

        "number_of_intervals":
            int(
                len(
                    intervals
                )
            ),

        "number_of_non_negative_intervals":
            int(
                len(
                    valid_waiting_times
                )
            ),

        "number_of_temporal_overlaps":
            int(
                number_of_overlaps
            ),

        "longest_non_negative_intervals":
            longest_waiting_times
            .to_dict(
                orient="records"
            )
    }


ANALYTICAL_FUNCTIONS = {

    "general":
        analyze_general_information,

    "activity_frequency":
        analyze_activity_frequency,

    "case_duration":
        analyze_case_duration,

    "variants":
        analyze_variants,

    "rework":
        analyze_rework,

    "case_endings":
        analyze_case_endings,

    "waiting_time":
        analyze_waiting_times
}


# =============================================================================
# Direct deterministic answers
# =============================================================================

def answer_with_python(question):

    q = normalize_text(
        question
    )


    quantity_query = (
        "how many" in q
        or
        "number of" in q
        or
        "total number of" in q
    )


    # Intensive rework

    if (
        quantity_query
        and
        "intensive rework" in q
    ):

        count = int(
            (
                case_base[
                    "rework_group"
                ]
                ==
                "Intensive rework"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "classified as intensive rework."
            ),
            "python_rework_intensive"
        )


    # Moderate rework

    if (
        quantity_query
        and
        "moderate rework" in q
    ):

        count = int(
            (
                case_base[
                    "rework_group"
                ]
                ==
                "Moderate rework"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "classified as moderate rework."
            ),
            "python_rework_moderate"
        )


    # No rework

    no_rework_patterns = [
        "no rework",
        "without rework",
        "do not present rework",
        "do not have rework",
        "have no rework"
    ]


    if (
        quantity_query
        and
        find_matches(
            q,
            no_rework_patterns
        )
    ):

        count = int(
            (
                case_base[
                    "rework_group"
                ]
                ==
                "No rework"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "without rework."
            ),
            "python_no_rework"
        )


    # Completed cases

    completed_case_patterns = [
        "completed case",
        "completed cases",
        "cases were completed",
        "cases are completed"
    ]

    if (
        quantity_query
        and
        find_matches(
            q,
            completed_case_patterns
        )
    ):

        count = int(
            (
                case_base[
                    "status"
                ]
                ==
                "Completed"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "classified as completed."
            ),
            "python_completed_cases"
        )


    # Cases terminated at RFQ

    if (
        quantity_query
        and
        (
            "terminated at rfq" in q
            or
            "ended at rfq" in q
        )
    ):

        count = int(
            (
                case_base[
                    "status"
                ]
                ==
                "Terminated at RFQ"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "terminated at RFQ."
            ),
            "python_rfq_endings"
        )


    # Cases terminated at Purchase Requisition

    if (
        quantity_query
        and
        (
            "terminated at purchase requisition"
            in q
            or
            "ended at purchase requisition"
            in q
        )
    ):

        count = int(
            (
                case_base[
                    "status"
                ]
                ==
                "Terminated at Purchase Requisition"
            )
            .sum()
        )

        return (
            (
                f"There are {count} cases "
                "terminated at Purchase Requisition."
            ),
            "python_requisition_endings"
        )


    # Specific variant

    variant_match = (
        re.search(
            r"\bv\s*(\d+)\b",
            q
        )
    )


    if (
        variant_match
        and
        quantity_query
    ):

        variant_number = int(
            variant_match
            .group(1)
        )

        variant_id = (
            f"V{variant_number}"
        )


        row = (
            variant_frequency[
                variant_frequency[
                    "variant_id"
                ]
                ==
                variant_id
            ]
        )


        if (
            len(row)
            ==
            0
        ):

            return (
                (
                    f"Variant {variant_id} "
                    "does not exist among "
                    "the variants identified "
                    "in the event log."
                ),
                "python_variant_not_found"
            )


        count = int(
            row
            .iloc[0][
                "case_count"
            ]
        )


        return (
            (
                f"Variant {variant_id} "
                f"contains {count} cases."
            ),
            "python_variant_case_count"
        )


    # Number of variants

    variant_count_patterns = [
        "how many variants",
        "how many process variants",
        "number of variants",
        "number of process variants",
        "total number of variants",
        "total number of process variants"
    ]

    if find_matches(
        q,
        variant_count_patterns
    ):

        return (
            (
                f"The event log contains "
                f"{number_of_variants} variants."
            ),
            "python_variant_count"
        )


    # Number of events

    if (
        "how many events" in q
        or
        "number of events" in q
        or
        "total number of events" in q
    ):

        return (
            (
                f"The event log contains "
                f"{number_of_events} events."
            ),
            "python_event_count"
        )


    # Number of activities

    activity_count_patterns = [
        "how many activities",
        "how many distinct activities",
        "number of activities",
        "number of distinct activities",
        "total number of activities",
        "total number of distinct activities"
    ]

    if find_matches(
        q,
        activity_count_patterns
    ):

        return (
            (
                f"The event log contains "
                f"{number_of_activities} "
                "distinct activities."
            ),
            "python_activity_count"
        )


    # Overall number of cases

    total_case_patterns = [
        "how many cases are there",
        "how many cases exist",
        "number of cases in the log",
        "total number of cases",
        "total cases"
    ]


    if (
        find_matches(
            q,
            total_case_patterns
        )
    ):

        return (
            (
                f"The event log contains "
                f"{number_of_cases} cases."
            ),
            "python_case_count"
        )


    # Main variant

    if (
        "main variant" in q
        or
        "most frequent variant" in q
    ):

        v1 = (
            variant_frequency
            .iloc[0]
        )


        return (
            (
                "The main variant is V1, "
                f"with {int(v1['case_count'])} "
                "cases.\n\n"
                "Flow:\n"
                f"{v1['variant']}"
            ),
            "python_main_variant"
        )


    # Overall median duration

    if (
        "median" in q
        and
        "duration" in q
        and
        "rework" not in q
        and
        "variant" not in q
        and
        "status" not in q
    ):

        median_hours = (
            case_base[
                "duration_hours"
            ]
            .median()
        )


        return (
            (
                "The overall median case duration "
                f"is {median_hours:.2f} hours "
                f"({median_hours / 24:.2f} days)."
            ),
            "python_median_duration"
        )


    # Overall mean duration

    if (
        "mean" in q
        and
        "duration" in q
        and
        "rework" not in q
        and
        "variant" not in q
        and
        "status" not in q
    ):

        mean_hours = (
            case_base[
                "duration_hours"
            ]
            .mean()
        )


        return (
            (
                "The overall mean case duration "
                f"is {mean_hours:.2f} hours "
                f"({mean_hours / 24:.2f} days)."
            ),
            "python_mean_duration"
        )


    return (
        None,
        None
    )


# =============================================================================
# Rule-based query routing
# =============================================================================

ROUTING_RULES = {

    "activity_frequency": [
        "activity frequency",
        "frequency of activities",
        "most frequent activity",
        "most frequent activities",
        "how often",
        "activity occurs",
        "activity appears"
    ],


    "case_duration": [
        "duration",
        "case duration",
        "case time",
        "longest cases",
        "slowest cases"
    ],


    "variants": [
        "variant",
        "variants",
        "flow",
        "happy path",
        "sequence of activities"
    ],


    "rework": [
        "rework",
        "amend",
        "amendment",
        "amendments",
        "modification",
        "modifications"
    ],


    "case_endings": [
        "ending",
        "endings",
        "ended",
        "terminated",
        "termination",
        "status",
        "final activity",
        "completed"
    ],


    "waiting_time": [
        "waiting",
        "waiting time",
        "time between activities",
        "delay between activities",
        "bottleneck"
    ],


    "general": [
        "process summary",
        "event log summary",
        "process overview",
        "general process information"
    ]
}


def select_contexts(question):

    q = normalize_text(
        question
    )


    identified_capabilities = []

    matched_keywords = []


    for (
        capability,
        keywords
    ) in (
        ROUTING_RULES.items()
    ):

        matches = (
            find_matches(
                q,
                keywords
            )
        )


        if matches:

            identified_capabilities.append(
                capability
            )

            matched_keywords.extend(
                matches
            )


    if (
        len(
            identified_capabilities
        )
        ==
        0
    ):

        return (
            None,
            [],
            []
        )


    context = {}


    for capability in (
        identified_capabilities
    ):

        analytical_function = (
            ANALYTICAL_FUNCTIONS[
                capability
            ]
        )

        context[
            capability
        ] = (
            analytical_function()
        )


    matched_keywords = (
        list(
            dict.fromkeys(
                matched_keywords
            )
        )
    )


    return (
        context,
        identified_capabilities,
        matched_keywords
    )


# =============================================================================
# LLM system instruction
# =============================================================================

SYSTEM_PROMPT = """
You are a Process Mining specialist.

Your task is to interpret only the analytical results
provided by the system.

Rules:

- Use only the analytical data provided.
- Do not invent numbers, facts, or relationships that are
  absent from the provided data.
- Do not use external knowledge to complete missing information.
- Do not answer in JSON.
- Be technical, objective, and clear.
- Cite numerical evidence when available.
- Distinguish observed facts from interpretations.
- If the evidence is insufficient, state this explicitly.
- If the data support only a partial interpretation,
  explicitly state the limitation.
- Do not establish causal relationships when the data
  only support association.
- When comparing groups, describe only differences observed
  in the provided data.
- Do not use the term "significantly" or equivalent expressions
  unless a statistical test result is provided.
- For bottleneck-related questions, treat high waiting times
  only as possible delay points, not as formally confirmed
  bottlenecks.
- When comparing durations, use the expressions
  "case duration" or "recorded case time".
- Do not use expressions that assume all cases were completed,
  because the event log contains different termination patterns.
"""


# =============================================================================
# Preprocessing summary
# =============================================================================

preprocessing_end = (
    time.perf_counter()
)


preprocessing_time = (
    preprocessing_end
    -
    preprocessing_start
)


print(
    "\nAnalytical base created."
)

print(
    "Preprocessing time:",
    round(
        preprocessing_time,
        4
    ),
    "seconds"
)


# =============================================================================
# Reproducibility metadata
# =============================================================================

experiment_configuration = {

    "architecture":
        "Hybrid",

    "model":
        GEMINI_MODEL,

    "temperature":
        TEMPERATURE,

    "event_log":
        EVENT_LOG_FILE,

    "top_n_variants":
        TOP_N_VARIANTS,

    "top_n_waiting_times":
        TOP_N_WAITING_TIMES,

    "rework_definition": {

        "activities":
            REWORK_ACTIVITIES,

        "classification": {

            "no_rework":
                "0 amendment activities",

            "moderate_rework":
                "1 or 2 amendment activities",

            "intensive_rework":
                "3 or more amendment activities"
        }
    },

    "routing_rules":
        ROUTING_RULES,

    "system_prompt":
        SYSTEM_PROMPT
}


with open(
    CONFIGURATION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        experiment_configuration,
        file,
        ensure_ascii=False,
        indent=2
    )


# =============================================================================
# Gemini client configuration
# =============================================================================

api_key = getpass(
    "Enter your Google AI Studio API key: "
)


client = genai.Client(
    api_key=api_key
)


GENERATION_CONFIG = (
    types.GenerateContentConfig(
        temperature=TEMPERATURE,
        system_instruction=SYSTEM_PROMPT
    )
)


print(
    f"Connected model: "
    f"{GEMINI_MODEL}"
)

print(
    f"Temperature: "
    f"{TEMPERATURE}"
)


# =============================================================================
# Token metadata
# =============================================================================

def extract_tokens(response):

    prompt_tokens = None

    output_tokens = None

    total_tokens = None


    try:

        usage = getattr(
            response,
            "usage_metadata",
            None
        )


        if (
            usage
            is not None
        ):

            prompt_tokens = getattr(
                usage,
                "prompt_token_count",
                None
            )


            output_tokens = getattr(
                usage,
                "candidates_token_count",
                None
            )


            total_tokens = getattr(
                usage,
                "total_token_count",
                None
            )


    except Exception:

        pass


    return (
        prompt_tokens,
        output_tokens,
        total_tokens
    )


# =============================================================================
# Experimental logging
# =============================================================================

experiment_records = []


def save_records():

    if (
        len(
            experiment_records
        )
        ==
        0
    ):
        return


    results_table = (
        pd.DataFrame(
            experiment_records
        )
    )


    results_table.to_csv(
        RESULTS_FILE,
        index=False,
        encoding="utf-8-sig"
    )


# =============================================================================
# Hybrid query execution
# =============================================================================

def ask_about_log(
    question,
    question_id=None
):

    start_timestamp = (
        datetime.now()
        .isoformat()
    )


    query_start = (
        time.perf_counter()
    )


    # Direct deterministic response

    (
        python_answer,
        python_route
    ) = (
        answer_with_python(
            question
        )
    )


    if (
        python_answer
        is not None
    ):

        total_latency = (
            time.perf_counter()
            -
            query_start
        )


        final_answer = (
            python_answer
            +
            "\n\n"
            +
            "Response source: "
            "direct Python analysis."
        )


        record = {

            "question_id":
                question_id,

            "timestamp":
                start_timestamp,

            "question":
                question,

            "architecture":
                "Hybrid",

            "response_type":
                "direct_python",

            "route":
                python_route,

            "used_llm":
                False,

            "matched_keywords":
                None,

            "selected_context":
                None,

            "context_payload":
                None,

            "response":
                final_answer,

            "total_latency_seconds":
                round(
                    total_latency,
                    6
                ),

            "llm_latency_seconds":
                None,

            "prompt_tokens":
                None,

            "output_tokens":
                None,

            "total_tokens":
                None,

            "rate_limit_error":
                False,

            "error":
                None
        }


        experiment_records.append(
            record
        )

        save_records()

        return final_answer


    # Rule-based routing

    (
        selected_context,
        capabilities,
        keywords
    ) = (
        select_contexts(
            question
        )
    )


    # Out-of-scope response

    if (
        selected_context
        is None
    ):

        total_latency = (
            time.perf_counter()
            -
            query_start
        )


        final_answer = (
            "The question does not match "
            "the analytical capabilities "
            "implemented in the prototype."
        )


        record = {

            "question_id":
                question_id,

            "timestamp":
                start_timestamp,

            "question":
                question,

            "architecture":
                "Hybrid",

            "response_type":
                "out_of_scope",

            "route":
                "out_of_scope",

            "used_llm":
                False,

            "matched_keywords":
                None,

            "selected_context":
                None,

            "context_payload":
                None,

            "response":
                final_answer,

            "total_latency_seconds":
                round(
                    total_latency,
                    6
                ),

            "llm_latency_seconds":
                None,

            "prompt_tokens":
                None,

            "output_tokens":
                None,

            "total_tokens":
                None,

            "rate_limit_error":
                False,

            "error":
                None
        }


        experiment_records.append(
            record
        )

        save_records()

        return final_answer


    # Structured analytical context

    context_json = (
        json.dumps(
            selected_context,
            ensure_ascii=False,
            indent=2,
            default=str
        )
    )


    user_prompt = f"""
SELECTED ANALYTICAL DATA:

{context_json}

USER QUESTION:

{question}
"""


    # LLM interpretation

    try:

        llm_start = (
            time.perf_counter()
        )


        response = (
            client
            .models
            .generate_content(
                model=GEMINI_MODEL,
                contents=user_prompt,
                config=GENERATION_CONFIG
            )
        )


        llm_end = (
            time.perf_counter()
        )


        llm_latency = (
            llm_end
            -
            llm_start
        )


        total_latency = (
            llm_end
            -
            query_start
        )


        (
            prompt_tokens,
            output_tokens,
            total_tokens
        ) = (
            extract_tokens(
                response
            )
        )


        response_text = getattr(
            response,
            "text",
            None
        )


        if (
            response_text
            is None
        ):

            raise ValueError(
                "The model did not return "
                "a textual response."
            )


        final_answer = (
            response_text
            +
            "\n\n"
            +
            "Response source: "
            "LLM interpretation based on "
            "selected analytical context."
        )


        record = {

            "question_id":
                question_id,

            "timestamp":
                start_timestamp,

            "question":
                question,

            "architecture":
                "Hybrid",

            "response_type":
                "llm_interpretation",

            "route":
                (
                    "llm_"
                    +
                    "+".join(
                        capabilities
                    )
                ),

            "used_llm":
                True,

            "matched_keywords":
                " | ".join(
                    keywords
                ),

            "selected_context":
                " | ".join(
                    capabilities
                ),

            "context_payload":
                context_json,

            "response":
                final_answer,

            "total_latency_seconds":
                round(
                    total_latency,
                    6
                ),

            "llm_latency_seconds":
                round(
                    llm_latency,
                    6
                ),

            "prompt_tokens":
                prompt_tokens,

            "output_tokens":
                output_tokens,

            "total_tokens":
                total_tokens,

            "rate_limit_error":
                False,

            "error":
                None
        }


        experiment_records.append(
            record
        )

        save_records()

        return final_answer


    except Exception as error:

        error_end = (
            time.perf_counter()
        )


        total_latency = (
            error_end
            -
            query_start
        )


        error_message = (
            str(error)
        )


        normalized_error = (
            error_message
            .lower()
        )


        is_rate_limit = (
            "429" in error_message
            or
            "resource_exhausted"
            in normalized_error
            or
            "rate limit"
            in normalized_error
        )


        record = {

            "question_id":
                question_id,

            "timestamp":
                start_timestamp,

            "question":
                question,

            "architecture":
                "Hybrid",

            "response_type":
                "llm_error",

            "route":
                (
                    "llm_"
                    +
                    "+".join(
                        capabilities
                    )
                ),

            "used_llm":
                True,

            "matched_keywords":
                " | ".join(
                    keywords
                ),

            "selected_context":
                " | ".join(
                    capabilities
                ),

            "context_payload":
                context_json,

            "response":
                None,

            "total_latency_seconds":
                round(
                    total_latency,
                    6
                ),

            "llm_latency_seconds":
                None,

            "prompt_tokens":
                None,

            "output_tokens":
                None,

            "total_tokens":
                None,

            "rate_limit_error":
                is_rate_limit,

            "error":
                error_message
        }


        experiment_records.append(
            record
        )

        save_records()

        raise


# =============================================================================
# Prototype summary
# =============================================================================

print(
    "\n--- HYBRID PROTOTYPE READY ---"
)

print(
    f"Cases: {number_of_cases}"
)

print(
    f"Events: {number_of_events}"
)

print(
    f"Distinct activities: "
    f"{number_of_activities}"
)

print(
    f"Variants: "
    f"{number_of_variants}"
)

print(
    f"Preprocessing time: "
    f"{preprocessing_time:.4f} seconds"
)

print(
    "The prototype is ready "
    "to receive questions."
)


# =============================================================================
# Interactive experiment interface
# =============================================================================

chat_history = []

last_llm_use = 0


question_box = (
    widgets.Textarea(

        placeholder=(
            "Enter your question "
            "about the event log..."
        ),

        description=(
            "Question:"
        ),

        layout=(
            widgets.Layout(
                width="100%",
                height="100px"
            )
        )
    )
)


ask_button = (
    widgets.Button(
        description="Ask",
        button_style="primary"
    )
)


clear_button = (
    widgets.Button(
        description="Clear history",
        button_style="warning"
    )
)


output = (
    widgets.Output()
)


def show_history():

    with output:

        output.clear_output()


        if (
            len(
                chat_history
            )
            ==
            0
        ):

            print(
                "No questions have "
                "been asked yet."
            )

            return


        display(
            Markdown(
                "# Conversation History"
            )
        )


        for (
            i,
            item
        ) in enumerate(
            chat_history,
            start=1
        ):

            display(
                Markdown(
                    f"### Question {i}\n"
                    f"**{item['question']}**"
                )
            )


            display(
                Markdown(
                    "**Answer:**\n\n"
                    f"{item['answer']}"
                )
            )


            display(
                Markdown("---")
            )


def on_ask_click(_):

    global last_llm_use


    question = (
        question_box
        .value
        .strip()
    )


    with output:

        if (
            question
            ==
            ""
        ):

            print(
                "Enter a question."
            )

            return


    (
        python_answer,
        _
    ) = (
        answer_with_python(
            question
        )
    )


    uses_llm = False


    if (
        python_answer
        is None
    ):

        (
            test_context,
            _,
            _
        ) = (
            select_contexts(
                question
            )
        )


        uses_llm = (
            test_context
            is not None
        )


    if (
        uses_llm
    ):

        now = (
            time.time()
        )


        elapsed_time = (
            now
            -
            last_llm_use
        )


        if (
            elapsed_time
            <
            MINIMUM_LLM_INTERVAL
        ):

            remaining = int(
                MINIMUM_LLM_INTERVAL
                -
                elapsed_time
            )


            with output:

                print(
                    "Please wait approximately "
                    f"{remaining} seconds "
                    "before the next "
                    "LLM request."
                )


            return


    try:

        ask_button.disabled = True

        ask_button.description = (
            "Processing..."
        )


        answer = (
            ask_about_log(
                question
            )
        )


        chat_history.append(
            {
                "question":
                    question,

                "answer":
                    answer
            }
        )


        if (
            uses_llm
        ):

            last_llm_use = (
                time.time()
            )


        question_box.value = ""


        show_history()


    except Exception as error:

        error_text = (
            str(error)
        )


        with output:

            if (
                "429"
                in error_text
                or
                "RESOURCE_EXHAUSTED"
                in error_text
            ):

                print(
                    "API rate limit reached. "
                    "The attempt was recorded "
                    "separately."
                )

            else:

                print(
                    "Error during request:"
                )

                print(
                    error_text
                )


    finally:

        ask_button.disabled = False

        ask_button.description = (
            "Ask"
        )


def clear_history(_):

    chat_history.clear()


    with output:

        output.clear_output()

        print(
            "Visual history cleared."
        )


ask_button.on_click(
    on_ask_click
)


clear_button.on_click(
    clear_history
)


display(
    question_box,
    widgets.HBox(
        [
            ask_button,
            clear_button
        ]
    ),
    output
)
